# 使用集合架构的聊天机器人

## 回顾

我们扩展了聊天机器人，将语义记忆保存到单个[用户档案](https://langchain-ai.github.io/langgraph/concepts/memory/#profile)中。

我们还介绍了一个库[Trustcall](https://github.com/hinthornw/trustcall)，用于使用新信息更新此架构。

## 目标

有时我们希望将记忆保存到[集合](https://docs.google.com/presentation/d/181mvjlgsnxudQI6S3ritg9sooNyu4AcLLFH1UK0kIuk/edit#slide=id.g30eb3c8cf10_0_200)而不是单个档案中。

在这里，我们将更新聊天机器人以[将记忆保存到集合](https://langchain-ai.github.io/langgraph/concepts/memory/#collection)中。

我们还将展示如何使用[Trustcall](https://github.com/hinthornw/trustcall)来更新此集合。

In [ ]:
%%capture --no-stderr
%pip install -U langchain_openai langgraph trustcall langchain_core

In [ ]:
import os, getpass

def _set_env(var: str):
    # 检查变量是否在OS环境中设置
    env_value = os.environ.get(var)
    if not env_value:
        # 如果未设置，提示用户输入
        env_value = getpass.getpass(f"{var}: ")
    
    # 为当前进程设置环境变量
    os.environ[var] = env_value

_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

## 定义集合架构

与将用户信息存储在固定的档案结构中不同，我们将创建一个灵活的集合架构来存储有关用户交互的记忆。

每个记忆将作为单独的条目存储，包含一个`content`字段用于存储我们想要记住的主要信息。

这种方法允许我们建立一个开放式的记忆集合，随着我们对用户了解的增加而增长和变化。

我们可以将集合架构定义为[Pydantic](https://docs.pydantic.dev/latest/)对象。

In [ ]:
from pydantic import BaseModel, Field

class Memory(BaseModel):
    content: str = Field(description="记忆的主要内容。例如：用户表达了对学习法语的兴趣。")

class MemoryCollection(BaseModel):
    memories: list[Memory] = Field(description="关于用户的记忆列表。")

In [ ]:
_set_env("OPENAI_API_KEY")

我们可以使用LangChain的[聊天模型](https://python.langchain.com/docs/concepts/chat_models/)接口的[`with_structured_output`](https://python.langchain.com/docs/concepts/structured_outputs/#recommended-usage)方法来强制结构化输出。

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

# 初始化模型
model = ChatOpenAI(model="gpt-4o", temperature=0)

# 将架构绑定到模型
model_with_structure = model.with_structured_output(MemoryCollection)

# 调用模型以生成与架构匹配的结构化输出
memory_collection = model_with_structure.invoke([HumanMessage("我叫Lance。我喜欢骑自行车。")])
memory_collection.memories

我们可以使用`model_dump()`将Pydantic模型实例序列化为Python字典。

In [ ]:
memory_collection.memories[0].model_dump()

将每个记忆的字典表示保存到存储中。

In [ ]:
import uuid
from langgraph.store.memory import InMemoryStore

# 初始化内存存储
in_memory_store = InMemoryStore()

# 要保存的记忆的命名空间
user_id = "1"
namespace_for_memory = (user_id, "memories")

# 将记忆保存到命名空间作为键和值
key = str(uuid.uuid4())
value = memory_collection.memories[0].model_dump()
in_memory_store.put(namespace_for_memory, key, value)

key = str(uuid.uuid4())
value = memory_collection.memories[1].model_dump()
in_memory_store.put(namespace_for_memory, key, value)

在存储中搜索记忆。

In [ ]:
# 搜索
for m in in_memory_store.search(namespace_for_memory):
    print(m.dict())

## 更新集合架构

我们在上一课中讨论了更新档案架构的挑战。

这同样适用于集合！

我们希望能够用新记忆更新集合，并更新集合中的现有记忆。

现在我们将展示[Trustcall](https://github.com/hinthornw/trustcall)也可以用于更新集合。

这使得既可以添加新记忆，也可以[更新集合中的现有记忆](https://github.com/hinthornw/trustcall?tab=readme-ov-file#simultanous-updates--insertions)。

让我们用Trustcall定义一个新的提取器。

和之前一样，我们提供每个记忆的架构`Memory`。

但是，我们可以提供`enable_inserts=True`来允许提取器向集合插入新记忆。

In [ ]:
from trustcall import create_extractor

# 创建提取器
trustcall_extractor = create_extractor(
    model,
    tools=[Memory],
    tool_choice="Memory",
    enable_inserts=True,
)

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# 指令
instruction = """从以下对话中提取记忆："""

# 对话
conversation = [HumanMessage(content="你好，我是Lance。"), 
                AIMessage(content="很高兴认识你，Lance。"), 
                HumanMessage(content="今天早上我在旧金山骑了一次很不错的自行车。")]

# 调用提取器
result = trustcall_extractor.invoke({"messages": [SystemMessage(content=instruction)] + conversation})

In [ ]:
# 消息包含工具调用
for m in result["messages"]:
    m.pretty_print()

In [ ]:
# 响应包含符合架构的记忆
for m in result["responses"]: 
    print(m)

In [ ]:
# 元数据包含工具调用
for m in result["response_metadata"]: 
    print(m)

In [ ]:
# 更新对话
updated_conversation = [AIMessage(content="太好了，之后你做了什么？"), 
                        HumanMessage(content="我去了Tartine，吃了一个羊角面包。"),                        
                        AIMessage(content="还有什么在你心里吗？"),
                        HumanMessage(content="我在想我的日本之行，今年冬天要再去一次！"),]

# 更新指令
system_msg = """根据以下对话更新现有记忆并创建新记忆："""

# 我们将保存现有记忆，给它们一个ID、键（工具名称）和值
tool_name = "Memory"
existing_memories = [(str(i), tool_name, memory.model_dump()) for i, memory in enumerate(result["responses"])] if result["responses"] else None
existing_memories

In [ ]:
# 使用我们更新的对话和现有记忆调用提取器
result = trustcall_extractor.invoke({"messages": updated_conversation, 
                                     "existing": existing_memories})

In [ ]:
# 来自模型的消息表明进行了两个工具调用
for m in result["messages"]:
    m.pretty_print()

In [ ]:
# 响应包含符合架构的记忆
for m in result["responses"]: 
    print(m)

这告诉我们，我们通过指定`json_doc_id`更新了集合中的第一个记忆。

In [ ]:
# 元数据包含工具调用
for m in result["response_metadata"]: 
    print(m)

LangSmith跟踪：

https://smith.langchain.com/public/ebc1cb01-f021-4794-80c0-c75d6ea90446/r

## 使用集合架构更新的聊天机器人

现在，让我们将Trustcall引入聊天机器人来创建和更新记忆集合。

In [ ]:
from IPython.display import Image, display

import uuid

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.store.memory import InMemoryStore
from langchain_core.messages import merge_message_runs
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables.config import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.base import BaseStore

# 初始化模型
model = ChatOpenAI(model="gpt-4o", temperature=0)

# 记忆架构
class Memory(BaseModel):
    content: str = Field(description="记忆的主要内容。例如：用户表达了对学习法语的兴趣。")

# 创建Trustcall提取器
trustcall_extractor = create_extractor(
    model,
    tools=[Memory],
    tool_choice="Memory",
    # 这允许提取器插入新记忆
    enable_inserts=True,
)

# 聊天机器人指令
MODEL_SYSTEM_MESSAGE = """您是一个有用的聊天机器人。您被设计为用户的伴侣。

您有一个长期记忆，跟踪您随时间了解的用户信息。

当前记忆（可能包括本次对话中的更新记忆）：

{memory}"""

# Trustcall指令
TRUSTCALL_INSTRUCTION = """反思以下互动。

使用提供的工具保留关于用户的任何必要记忆。

使用并行工具调用同时处理更新和插入："""

def call_model(state: MessagesState, config: RunnableConfig, store: BaseStore):

    """从存储中加载记忆并使用它们个性化聊天机器人的响应。"""
    
    # 从配置中获取用户ID
    user_id = config["configurable"]["user_id"]

    # 从存储中检索记忆
    namespace = ("memories", user_id)
    memories = store.search(namespace)

    # 为系统提示格式化记忆
    info = "\n".join(f"- {mem.value['content']}" for mem in memories)
    system_msg = MODEL_SYSTEM_MESSAGE.format(memory=info)

    # 使用记忆和聊天历史进行响应
    response = model.invoke([SystemMessage(content=system_msg)]+state["messages"])

    return {"messages": response}

def write_memory(state: MessagesState, config: RunnableConfig, store: BaseStore):

    """反思聊天历史并更新记忆集合。"""
    
    # 从配置中获取用户ID
    user_id = config["configurable"]["user_id"]

    # 定义记忆的命名空间
    namespace = ("memories", user_id)

    # 检索最新的记忆以供上下文使用
    existing_items = store.search(namespace)

    # 为Trustcall提取器格式化现有记忆
    tool_name = "Memory"
    existing_memories = ([(existing_item.key, tool_name, existing_item.value)
                          for existing_item in existing_items]
                          if existing_items
                          else None
                        )

    # 合并聊天历史和指令
    updated_messages=list(merge_message_runs(messages=[SystemMessage(content=TRUSTCALL_INSTRUCTION)] + state["messages"]))

    # 调用提取器
    result = trustcall_extractor.invoke({"messages": updated_messages, 
                                        "existing": existing_memories})

    # 将Trustcall的记忆保存到存储中
    for r, rmeta in zip(result["responses"], result["response_metadata"]):
        store.put(namespace,
                  rmeta.get("json_doc_id", str(uuid.uuid4())),
                  r.model_dump(mode="json"),
            )

# 定义图
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_node("write_memory", write_memory)
builder.add_edge(START, "call_model")
builder.add_edge("call_model", "write_memory")
builder.add_edge("write_memory", END)

# 长期（跨线程）记忆的存储
across_thread_memory = InMemoryStore()

# 短期（线程内）记忆的检查点
within_thread_memory = MemorySaver()

# 使用检查点和存储编译图
graph = builder.compile(checkpointer=within_thread_memory, store=across_thread_memory)

# 查看
display(Image(graph.get_graph(xray=1).draw_mermaid_png()))

In [ ]:
# 我们为短期（线程内）记忆提供线程ID
# 我们为长期（跨线程）记忆提供用户ID
config = {"configurable": {"thread_id": "1", "user_id": "1"}}

# 用户输入
input_messages = [HumanMessage(content="你好，我叫Lance")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

In [ ]:
# 用户输入
input_messages = [HumanMessage(content="我喜欢在旧金山骑自行车")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

In [ ]:
# 要保存的记忆的命名空间
user_id = "1"
namespace = ("memories", user_id)
memories = across_thread_memory.search(namespace)
for m in memories:
    print(m.dict())

In [ ]:
# 用户输入
input_messages = [HumanMessage(content="我还喜欢去面包店")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

在新线程中继续对话。

In [ ]:
# 我们为短期（线程内）记忆提供线程ID
# 我们为长期（跨线程）记忆提供用户ID
config = {"configurable": {"thread_id": "2", "user_id": "1"}}

# 用户输入
input_messages = [HumanMessage(content="你推荐我去哪些面包店？")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

### LangSmith

https://smith.langchain.com/public/c87543ec-b426-4a82-a3ab-94d01c01d9f4/r

## Studio

![Screenshot 2024-10-30 at 11.29.25 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/6732d0876d3daa19fef993ba_Screenshot%202024-11-11%20at%207.50.21%E2%80%AFPM.png)